# 01 - Correccion de senales sismicas

Caso USGS/PRISM: lectura SMC sin corregir, baseline, despiking, filtro, integracion y comparacion contra acelerograma corregido de referencia.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from signalprocessor.io import read_motion
from signalprocessor.processing import CorrectionConfig, correct_record
from signalprocessor.metrics import compute_ground_motion_parameters
from signalprocessor.spectra import response_spectrum


In [ ]:
raw_path = ROOT / 'examples/data/benchmark/uncorrected_motion/CCSP.HNN.._u.smc'
ref_path = ROOT / 'examples/data/benchmark/corrected_motion/CCSP.HNN.._a.smc'

raw = read_motion(raw_path)
reference = read_motion(ref_path, units='cm/s^2')

cfg = CorrectionConfig(
    baseline_order=1,
    constrain_final_velocity=True,
    constrain_final_displacement=False,
    highpass_hz=0.05,
    lowpass_hz=25.0,
    filter_order=4,
)
result = correct_record(raw, cfg)
ours = result.record.as_units('cm/s^2')

params = pd.DataFrame([
    compute_ground_motion_parameters(raw).__dict__ | {'stage': 'raw'},
    result.metrics.__dict__ | {'stage': 'corrected'},
    compute_ground_motion_parameters(reference).__dict__ | {'stage': 'usgs_ref'},
]).set_index('stage')
params[['pga', 'pgv', 'pgd', 'arias_intensity', 'd5_95', 'final_velocity', 'final_displacement']]


In [ ]:
n = min(ours.npts, reference.npts)
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(raw.time, raw.as_units('cm/s^2').acceleration, lw=0.8, label='raw')
axes[0].plot(ours.time[:n], ours.acceleration[:n], lw=0.9, label='python/cython')
axes[0].plot(reference.time[:n], reference.acceleration[:n], lw=0.7, alpha=0.7, label='USGS')
axes[0].set_ylabel('Acc (cm/s2)')
axes[0].legend()

axes[1].plot(ours.time, result.velocity, lw=0.9)
axes[1].set_ylabel('Vel (m/s)')

axes[2].plot(ours.time, result.displacement, lw=0.9)
axes[2].set_ylabel('Disp (m)')
axes[2].set_xlabel('Time (s)')
fig.tight_layout()


In [ ]:
periods = np.geomspace(0.02, 5.0, 80)
spec_ours = response_spectrum(ours, periods)
spec_ref = response_spectrum(reference, periods)

plt.figure(figsize=(8, 5))
plt.loglog(spec_ours.periods, spec_ours.sa, label='python/cython')
plt.loglog(spec_ref.periods, spec_ref.sa, label='USGS')
plt.xlabel('T (s)')
plt.ylabel('PSA (g)')
plt.grid(True, which='both', alpha=0.3)
plt.legend();
